# Finaler Lauf & Vergleich

Dieses Notebook **zeigt** die getunten Ergebnisse und die finalen Metriken — das
eigentliche (stundenlange) Training läuft bewusst **per Skript**, nicht hier
(robuster bei langen Läufen; TensorBoard läuft separat).

Ablauf:
1. Getunte Params aus der Optuna-Study anzeigen
2. Optimization-History der Trials
3. Finales Training per Skript starten (Befehle unten)
4. Finale Ergebnisse, Lernkurven und Signifikanzvergleich

> Alle Zellen sind gegen fehlende Dateien abgesichert — du kannst "Run All"
> auch dann drücken, wenn erst DQN (und noch nicht PPO) fertig ist.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from lunarlander import config

ALGOS = ["dqn", "ppo"]

## 1. Getunte Ergebnisse
Beste Rendite und Hyperparameter je Algorithmus — direkt aus der Study-DB, nichts abtippen.

In [ ]:
studies = {}
for algo in ALGOS:
    db = config.STUDIES_DIR / f"{algo}.db"
    if db.exists():
        s = optuna.load_study(study_name=f"{algo}_lunarlander", storage=f"sqlite:///{db}")
        studies[algo] = s
        n_pruned = sum(t.state.name == "PRUNED" for t in s.trials)
        print(f"[{algo.upper()}]  {len(s.trials)} Trials ({n_pruned} gepruned)  |  beste Rendite: {s.best_value:.1f}")
        print(f"   beste Params: {s.best_params}\n")
    else:
        print(f"[{algo.upper()}]  noch keine Study ({db} fehlt)\n")

## 2. Optimization-History
Verlauf der Trial-Werte — steigt die beste Rendite über die Trials an?

In [ ]:
import optuna.visualization.matplotlib as vm
import matplotlib.pyplot as plt

for algo, s in studies.items():
    ax = vm.plot_optimization_history(s)
    ax.set_title(f"{algo.upper()} — Optimization History")
    plt.show()

## 3. Finales Training starten (im Terminal)
Nicht hier ausführen — in einem Terminal laufen lassen. Das Skript liest die
besten Params automatisch aus der Study (kein `--params` nötig) und schreibt
`results/metrics/<algo>.csv` sowie TensorBoard-Logs.

```bash
uv run python -m scripts.run_final_eval --algo dqn
uv run python -m scripts.run_final_eval --algo ppo

# Live beobachten (zweites Terminal):
uv run tensorboard --logdir results/tb      # -> http://localhost:6006
```

Wenn beide Läufe fertig sind, unten weiter mit "Run All".

## 4. Finale Ergebnisse
Rendite je Seed aus `results/metrics/<algo>.csv`.

In [ ]:
import pandas as pd

metrics = {}
for algo in ALGOS:
    csv = config.METRICS_DIR / f"{algo}.csv"
    if csv.exists():
        df = pd.read_csv(csv)
        metrics[algo] = df["mean_reward"].values
        print(f"[{algo.upper()}] je Seed: {df['mean_reward'].round(1).tolist()}")
        print(f"   Mittel: {df['mean_reward'].mean():.1f}\n")
    else:
        print(f"[{algo.upper()}] noch keine Metriken ({csv} fehlt)\n")

## 5. Lernkurven & Signifikanzvergleich
Lernkurven (±Std über Seeds), Boxplot und der Welch-Test (p-Wert, Konfidenz-
intervalle, Cohen's d) — die Kernaussagen fürs Poster.

In [ ]:
from lunarlander import plots, stats
from IPython.display import Image, display

# Lernkurven aus den finalen Läufen (evaluations.npz je Seed), sofern vorhanden
paths = {}
for algo in ALGOS:
    seed_paths = [config.MODELS_DIR / f"{algo}_seed{s}" / "evaluations.npz" for s in config.SEEDS]
    if all(p.exists() for p in seed_paths):
        paths[algo] = seed_paths

if paths:
    png = plots.learning_curve(
        paths, config.RESULTS_DIR / "final_learning_curves.png",
        mark_best_checkpoints=True,
    )
    display(Image(filename=str(png)))
else:
    print("Noch keine finalen Lernkurven-Logs vorhanden.")

In [ ]:
# Vergleich + Signifikanz, sobald beide Algorithmen fertig sind
if "dqn" in metrics and "ppo" in metrics:
    cmp_png = plots.comparison_plot({"dqn": metrics["dqn"], "ppo": metrics["ppo"]},
                                    config.RESULTS_DIR / "final_comparison.png")
    display(Image(filename=str(cmp_png)))

    result = stats.summarize(metrics["dqn"], metrics["ppo"], "dqn", "ppo")
    print("Signifikanzvergleich DQN vs. PPO:")
    for k, v in result.items():
        print(f"  {k}: {v}")
else:
    print("Vergleich sobald DQN und PPO fertig sind (beide metrics/*.csv vorhanden).")

## 6. Demo-GIF vom besten Seed
Für ein **vorzeigbares Artefakt** (nicht für die Statistik!): wählt automatisch den
Seed mit der höchsten Rendite aus der CSV, lädt dessen bestes Modell und rendert eine
Episode als GIF. Bewusst als „bestes von N Seeds" zu verstehen — die *berichtete*
Leistung bleibt der Mittelwert über alle Seeds (Abschnitt 4/5).

In [ ]:
import imageio
from stable_baselines3 import DQN, PPO
from lunarlander.envs import make_env

DEMO_ALGO = "dqn"   # Algorithmus fürs Demo-GIF ("dqn" oder "ppo")

csv = config.METRICS_DIR / f"{DEMO_ALGO}.csv"
if not csv.exists():
    print(f"Noch keine Metriken für {DEMO_ALGO} — erst den finalen Lauf machen.")
else:
    # Besten Seed aus der CSV wählen (höchste mittlere Rendite)
    df = pd.read_csv(csv)
    best = df.loc[df["mean_reward"].idxmax()]
    best_seed = int(best["seed"])
    print(f"Bester {DEMO_ALGO.upper()}-Seed: {best_seed}  (mean_reward={best['mean_reward']:.1f})")

    # Bester Snapshot dieses Seeds (best_model vom EvalCallback), sonst final_model
    model_dir = config.MODELS_DIR / f"{DEMO_ALGO}_seed{best_seed}"
    model_path = model_dir / "best_model"
    if not model_path.with_suffix(".zip").exists():
        model_path = model_dir / "final_model"
    model = {"dqn": DQN, "ppo": PPO}[DEMO_ALGO].load(model_path)

    # Eine Episode deterministisch rendern
    render_env = make_env(seed=best_seed, render_mode="rgb_array")
    obs, _ = render_env.reset(seed=best_seed)
    frames, total, done = [], 0.0, False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, term, trunc, _ = render_env.step(int(action))
        frames.append(render_env.render())
        total += reward
        done = term or trunc
    render_env.close()

    gif_path = config.RESULTS_DIR / f"demo_{DEMO_ALGO}_best_seed{best_seed}.gif"
    imageio.mimsave(gif_path, frames, fps=30)
    print(f"Episode-Rendite: {total:.1f}  |  {len(frames)} Frames -> {gif_path}")
    display(Image(filename=str(gif_path)))